In [1]:
import open3d as o3d
import numpy as np

def run_high_quality_reconstruction():
    mesh_file = "scene.gltf"
    print("1. Loading source 3D model...")
    mesh = o3d.io.read_triangle_mesh(mesh_file)
    mesh.compute_vertex_normals()

    # STEP 1: INCREASE DENSITY (200,000 points for crisp detail)
    print("2. Sampling high-density point cloud (100,000 points)...")
    pcd = mesh.sample_points_poisson_disk(number_of_points=100000)
    o3d.io.write_point_cloud("high_res_pointcloud.ply", pcd)

    # STEP 2: NORMALS ESTIMATION WITH OPTIMIZED NEIGHBORHOOD
    print("3. Computing precise surface normals...")
    pcd.estimate_normals(
        search_param=o3d.geometry.KDTreeSearchParamHybrid(radius=0.02, max_nn=50)
    )
    pcd.orient_normals_consistent_tangent_plane(k=15)

    # STEP 3: HIGH-RESOLUTION POISSON RECONSTRUCTION (Depth 10)
    print("4. Running High-Res Poisson Surface Reconstruction (Depth=10)...")
    reconstructed_mesh, densities = o3d.geometry.TriangleMesh.create_from_point_cloud_poisson(
        pcd, depth=10, scale=1.1
    )

    # Clean up outlier geometry
    vertices_to_remove = densities < np.quantile(densities, 0.01)
    reconstructed_mesh.remove_vertices_by_mask(vertices_to_remove)
    reconstructed_mesh.compute_vertex_normals()

    # Save output mesh
    output_mesh_file = "high_res_reconstructed_warehouse.obj"
    o3d.io.write_triangle_mesh(output_mesh_file, reconstructed_mesh)
    print(f"Saved high-quality mesh to: {output_mesh_file}")

    # STEP 4: CREATE A SLICED/CUTAWAY VERSION TO SEE INSIDE
    print("5. Generating cutaway mesh for interior inspection...")
    bbox = reconstructed_mesh.get_axis_aligned_bounding_box()
    extent = bbox.get_extent()
    max_bound = bbox.get_max_bound()
    min_bound = bbox.get_min_bound()

    # Clip off the upper 30% (roof) to allow interior view
    max_bound[1] -= extent[1] * 0.30  # Adjust axis index (0=X, 1=Y, 2=Z) if needed
    crop_box = o3d.geometry.AxisAlignedBoundingBox(min_bound, max_bound)
    interior_cutaway_mesh = reconstructed_mesh.crop(crop_box)

    # STEP 5: VISUALIZE
    print("\n--- VISUALIZATION CONTROLS ---")
    print(" * Press 'W' to toggle Wireframe (see through walls)")
    print(" * Press 'L' to toggle interior lighting")
    print(" * Shift + Mouse Drag to move inside rooms")
    print("-------------------------------\n")

    print("Displaying Cutaway Interior View...")
    o3d.visualization.draw_geometries(
        [interior_cutaway_mesh], 
        window_name="Interior Inspection View (Roof Cutaway)"
    )

    print("Displaying Full High-Res Mesh...")
    o3d.visualization.draw_geometries(
        [reconstructed_mesh], 
        window_name="Full High-Res Reconstructed Mesh"
    )

if __name__ == "__main__":
    run_high_quality_reconstruction()

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.
1. Loading source 3D model...
2. Sampling high-density point cloud (100,000 points)...
3. Computing precise surface normals...
4. Running High-Res Poisson Surface Reconstruction (Depth=10)...
[Open3D WARNING] Write OBJ can not include triangle normals.


[WARNING] /root/Open3D/build/poisson/src/ext_poisson/PoissonRecon/Src/FEMTree.IsoSurface.specialized.inl (Line 1858)
          Extract
          bad average roots: 2


Saved high-quality mesh to: high_res_reconstructed_warehouse.obj
5. Generating cutaway mesh for interior inspection...

--- VISUALIZATION CONTROLS ---
 * Press 'W' to toggle Wireframe (see through walls)
 * Press 'L' to toggle interior lighting
 * Shift + Mouse Drag to move inside rooms
-------------------------------

Displaying Cutaway Interior View...
Displaying Full High-Res Mesh...


points smoothing 


In [ ]:
import open3d as o3d
import numpy as np

def generate_clean_architecture_mesh(gltf_or_ply_path, output_name="clean_house"):
    print("1. Loading model...")
    if gltf_or_ply_path.endswith(".gltf") or gltf_or_ply_path.endswith(".glb"):
        mesh_in = o3d.io.read_triangle_mesh(gltf_or_ply_path)
        # Sample points with vertex colors intact
        pcd = mesh_in.sample_points_poisson_disk(number_of_points=150000)
    else:
        pcd = o3d.io.read_point_cloud(gltf_or_ply_path)

    print("2. Estimating sharp architectural normals...")
    # Smaller radius keeps wall corners flat and sharp
    pcd.estimate_normals(
        search_param=o3d.geometry.KDTreeSearchParamHybrid(radius=0.01, max_nn=30)
    )
    pcd.orient_normals_consistent_tangent_plane(k=15)

    print("3. Running Ball Pivoting Reconstruction (Sharp Wall Surfaces)...")
    # Radii define the size of rolling balls to connect points without bloating
    distances = pcd.compute_nearest_neighbor_distance()
    avg_dist = np.mean(distances)
    radii = [avg_dist * 0.5, avg_dist, avg_dist * 2, avg_dist * 4]

    rec_mesh = o3d.geometry.TriangleMesh.create_from_point_cloud_ball_pivoting(
        pcd, o3d.utility.DoubleVector(radii)
    )

    print("4. Cleaning geometry...")
    rec_mesh.remove_degenerate_triangles()
    rec_mesh.remove_duplicated_triangles()
    rec_mesh.remove_duplicated_vertices()
    rec_mesh.remove_non_manifold_edges()
    rec_mesh.compute_vertex_normals()

    # Save outputs
    ply_out = f"{output_name}.ply"
    obj_out = f"{output_name}.obj"
    
    o3d.io.write_triangle_mesh(ply_out, rec_mesh)
    o3d.io.write_triangle_mesh(obj_out, rec_mesh)
    
    print(f"Done! Saved sharp meshes: '{ply_out}' and '{obj_out}'")

    # Render in Open3D
    o3d.visualization.draw_geometries([rec_mesh], window_name="Clean Sharp Architecture")

if __name__ == "__main__":
    generate_clean_architecture_mesh("scene.gltf", output_name="sharp_vacation_house")

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.
1. Loading model...
2. Estimating sharp architectural normals...
3. Running Ball Pivoting Reconstruction (Sharp Wall Surfaces)...
4. Cleaning geometry...
[Open3D WARNING] Write OBJ can not include triangle normals.
Done! Saved sharp meshes: 'sharp_vacation_house.ply' and 'sharp_vacation_house.obj'
